# Bolão Copa do Mundo FIFA 2026 — Previsão de placares com Rede Neural (TensorFlow/Keras)

**Aluno:** Bruno Aires &nbsp;|&nbsp; **Turma:** 2º BIM 2026

Pipeline reprodutível que treina uma **rede neural de regressão de Poisson** para prever os
placares exatos dos **24 jogos da 1ª rodada da fase de grupos** da Copa do Mundo 2026 e exporta
o resultado em JSON no formato exigido por `bolao_copa.txt`.

## 1. Introdução

**Hipótese do modelo.** O número de gols marcados por cada seleção em uma partida é uma
**contagem discreta não-negativa** que pode ser modelada por uma distribuição de **Poisson**,
cujo parâmetro de intensidade $\lambda$ (gols esperados) é uma **função não-linear** da força
relativa entre as equipes. Essa força é capturada por: rating **Elo** (construído
cronologicamente sobre 20 anos de jogos), **ranking/pontos FIFA**, **forma recente**
(gols pró/contra e aproveitamento nos últimos jogos) e a **probabilidade implícita do mercado de
apostas** (odds 1/X/2 convertidas). A vantagem de mando de campo **não é ajustada manualmente**:
fornecemos apenas a *feature* `neutral` (e o status de país-sede) e deixamos a rede aprender o efeito.

**Correção de Data Leakage.** Para evitar que a rede neural sofra de vazamento de dados ao mapear
o Elo diretamente como proxy perfeito das odds no treino, as probabilidades históricas implícitas
foram calibradas adicionando ruído via **Distribuição de Dirichlet**. Isso mimetiza a incerteza e o
spread real das casas de apostas no passado, permitindo que o modelo generalize corretamente ao
receber as cotações reais trazidas de `Copa_do_Mundo_Cotacoes.pdf` para os jogos de 2026.

**Abordagem.** Uma rede densa recebe o vetor de atributos do confronto e produz **dois valores**
$(\lambda_A, \lambda_B)$ — gols esperados de cada seleção — via ativação `softplus` (garante
$\lambda>0$). O treinamento minimiza a **log-verossimilhança negativa de Poisson**. Cada partida
histórica é ponderada por **decay temporal** (ênfase nos últimos 2 anos). Para seleções com
histórico esparso aplicamos **transfer learning** por confederação (encolhimento bayesiano
das estatísticas em direção à média da confederação). A previsão final de cada jogo é um
**placar inteiro determinístico**: a **moda** da Poisson de cada $\lambda$, restrita ao intervalo
histórico realista $[0,5]$. **Todas as previsões derivam exclusivamente da rede neural**.

### Configuração e reprodutibilidade

Fixamos *seeds* em `PYTHONHASHSEED`, `random`, `numpy` e `tensorflow`, e habilitamos operações
determinísticas no TF. Assim, execuções subsequentes do notebook produzem **exatamente os mesmos
resultados**.

In [3]:
# Imports e configuração global de reprodutibilidade.
import os, random, json, io, warnings
SEED = 42                                       # semente única usada em todo o pipeline
os.environ["PYTHONHASHSEED"] = str(SEED)
os.environ["TF_DETERMINISTIC_OPS"] = "1"
os.environ["TF_CUDNN_DETERMINISTIC"] = "1"
os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "2")
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow import keras
from scipy.stats import poisson, dirichlet       # Adicionado dirichlet para calibração de odds

random.seed(SEED)
np.random.seed(SEED)
tf.keras.utils.set_random_seed(SEED)
try:
    tf.config.experimental.enable_op_determinism()
except Exception as e:
    print("determinism aviso:", e)

print("TensorFlow:", tf.__version__, "| NumPy:", np.__version__, "| Pandas:", pd.__version__)

TensorFlow: 2.20.0 | NumPy: 2.0.2 | Pandas: 2.2.2


## 2. Calendário e dados dos jogos

Coletamos o calendário oficial da 1ª rodada da fase de grupos (datas, horários, sedes, grupos e
pareamentos). O notebook recorre à tabela embutida corrigida conforme chaves oficiais da FIFA de 2026.

In [4]:
FIXTURES = [
    # jogo, sigla_A, sigla_B, data,        hora,    grupo, A_é_sede
    (1,  "MEX","RSA","2026-06-11","16:00","A", True ),
    (2,  "KOR","CZE","2026-06-11","23:00","A", False),
    (3,  "CAN","BIH","2026-06-12","16:00","B", True ),
    (4,  "USA","PAR","2026-06-12","22:00","D", True ),
    (5,  "HAI","SCO","2026-06-13","22:00","C", False),
    (6,  "AUS","TUR","2026-06-14","01:00","D", False),
    (7,  "BRA","MAR","2026-06-13","19:00","C", False),
    (8,  "QAT","SUI","2026-06-13","16:00","B", False),
    (9,  "CIV","ECU","2026-06-14","13:00","E", False),
    (10, "GER","CUW","2026-06-14","14:00","E", False),
    (11, "NED","JPN","2026-06-14","17:00","F", False),
    (12, "SWE","TUN","2026-06-14","23:00","F", False),
    (13, "KSA","URU","2026-06-15","19:00","H", False),
    (14, "ESP","CPV","2026-06-15","13:00","H", False),
    (15, "IRN","NZL","2026-06-15","22:00","G", False),
    (16, "BEL","EGY","2026-06-15","16:00","G", False),
    (17, "FRA","SEN","2026-06-16","16:00","I", False),
    (18, "IRQ","NOR","2026-06-16","19:00","I", False),
    (19, "ARG","ALG","2026-06-16","22:00","J", False),
    (20, "AUT","JOR","2026-06-17","01:00","J", False),
    (21, "GHA","PAN","2026-06-17","20:00","L", False),
    (22, "ENG","CRO","2026-06-17","17:00","L", False),
    (23, "POR","COD","2026-06-17","14:00","K", False),
    (24, "UZB","COL","2026-06-17","23:00","K", False),
]

import requests
def fetch_calendar():
    try:
        url = "https://en.wikipedia.org/wiki/2026_FIFA_World_Cup_group_stage"
        r = requests.get(url, timeout=8, headers={"User-Agent":"Mozilla/5.0"})
        r.raise_for_status()
        return r.text
    except Exception as e:
        print("[calendario] fetch online indisponivel -> usando tabela embutida.")
        return None

_html = fetch_calendar()
cal = pd.DataFrame(FIXTURES, columns=["jogo","A","B","data","hora","grupo","A_sede"])
cal["confronto"] = cal["A"] + " x " + cal["B"]
print(f"{len(cal)} jogos da 1a rodada carregados.")

[calendario] fetch online indisponivel -> usando tabela embutida.
24 jogos da 1a rodada carregados.


## 3. Fontes de dados e Odds

In [5]:
# Odds reais coletadas de Copa_do_Mundo_Cotacoes.pdf para inferência em 2026
ODDS = {
    1:(1.50,4.25,6.90),  2:(2.75,3.10,2.75),  3:(1.83,3.65,4.40),  4:(2.05,3.35,3.75),
    5:(6.60,4.35,1.50),  6:(4.40,3.80,1.78),  7:(1.62,3.80,5.80),  8:(10.75,5.60,1.28),
    9:(3.85,2.85,2.30),  10:(1.04,14.50,55.00), 11:(1.98,3.75,3.60), 12:(1.95,3.35,4.15),
    13:(6.50,4.35,1.50), 14:(1.09,10.00,30.00), 15:(1.93,3.45,4.15), 16:(1.67,3.85,5.10),
    17:(1.47,4.40,6.80), 18:(11.00,5.90,1.26), 19:(1.42,4.50,8.25), 20:(1.37,5.20,8.00),
    21:(1.98,3.55,3.80), 22:(1.78,3.70,4.55), 23:(1.28,5.70,10.75), 24:(7.90,4.50,1.42),
}
def odds_to_prob(o1, ox, o2):
    inv = np.array([1.0/o1, 1.0/ox, 1.0/o2])
    return inv / inv.sum()

imp = {j: odds_to_prob(*o) for j, o in ODDS.items()}
cal[["imp_1","imp_x","imp_2"]] = np.vstack([imp[int(j)] for j in cal["jogo"]])

In [8]:
DATA_DIR = "data"
RESULTS_URL = "https://raw.githubusercontent.com/martj42/international_results/master/results.csv"

def load_results():
    try:
        resp = requests.get(RESULTS_URL, timeout=15)
        resp.raise_for_status()
        df = pd.read_csv(io.StringIO(resp.text), parse_dates=["date"])
        print("[results] baixado online:", df.shape)
    except Exception as e:
        df = pd.read_csv(os.path.join(DATA_DIR, "results.csv"), parse_dates=["date"])
        print("[results] local:", df.shape)
    return df

raw = load_results()
fifa = pd.read_csv(os.path.join(DATA_DIR, "fifa_ranking.csv"), parse_dates=["rank_date"])
print("[fifa_ranking]:", fifa.shape)

[results] baixado online: (49437, 9)
[fifa_ranking]: (62424, 9)


## 4. Pré-processamento e Modelagem de Odds Históricas

In [9]:
FIFA_RENAME = {"Korea Republic":"South Korea","USA":"United States","Côte d'Ivoire":"Ivory Coast",
               "Cabo Verde":"Cape Verde","IR Iran":"Iran","Congo DR":"DR Congo","China PR":"China",
               "Kyrgyz Republic":"Kyrgyzstan","Cape Verde Islands":"Cape Verde"}
fifa["country_full"] = fifa["country_full"].replace(FIFA_RENAME)

ABRV2NAME = {"MEX":"Mexico","RSA":"South Africa","KOR":"South Korea","CZE":"Czech Republic",
    "CAN":"Canada","BIH":"Bosnia and Herzegovina","USA":"United States","PAR":"Paraguay",
    "HAI":"Haiti","SCO":"Scotland","AUS":"Australia","TUR":"Turkey","BRA":"Brazil","MAR":"Morocco",
    "QAT":"Qatar","SUI":"Switzerland","CIV":"Ivory Coast","ECU":"Ecuador","GER":"Germany",
    "CUW":"Curaçao","NED":"Netherlands","JPN":"Japan","SWE":"Sweden","TUN":"Tunisia",
    "KSA":"Saudi Arabia","URU":"Uruguay","ESP":"Spain","CPV":"Cape Verde","IRN":"Iran",
    "NZL":"New Zealand","BEL":"Belgium","EGY":"Egypt","FRA":"France","SEN":"Senegal","IRQ":"Iraq",
    "NOR":"Norway","ARG":"Argentina","ALG":"Algeria","AUT":"Austria","JOR":"Jordan","GHA":"Ghana",
    "PAN":"Panama","ENG":"England","CRO":"Croatia","POR":"Portugal","COD":"DR Congo",
    "UZB":"Uzbekistan","COL":"Colombia"}

name2conf = (fifa.sort_values("rank_date").drop_duplicates("country_full", keep="last")
                 .set_index("country_full")["confederation"].to_dict())
def conf_of(name): return name2conf.get(name, "OTHER")

df = raw.dropna(subset=["home_score","away_score"]).copy()
df["home_score"] = df["home_score"].astype(int)
df["away_score"] = df["away_score"].astype(int)
df = df.sort_values("date").reset_index(drop=True)
DATA_MAX = df["date"].max()

In [10]:
from collections import deque, defaultdict
K_FORM, ELO_BASE, ELO_K = 6, 1500.0, 30.0
elo = defaultdict(lambda: ELO_BASE)
form = defaultdict(lambda: deque(maxlen=K_FORM))
cnt  = defaultdict(int)

def form_stats(t):
    dq = form[t]
    if not dq: return (np.nan, np.nan, np.nan)
    a = np.asarray(dq, float)
    return (a[:,0].mean(), a[:,1].mean(), a[:,2].mean())

cols = {k: [] for k in ["elo_h","elo_a","f_gf_h","f_ga_h","f_ppg_h","f_gf_a","f_ga_a","f_ppg_a"]}
for h, a, hs, asc in zip(df["home_team"], df["away_team"], df["home_score"], df["away_score"]):
    eh, ea = elo[h], elo[a]
    gfh, gah, pph = form_stats(h); gfa, gaa, ppa = form_stats(a)
    cols["elo_h"].append(eh); cols["elo_a"].append(ea)
    cols["f_gf_h"].append(gfh); cols["f_ga_h"].append(gah); cols["f_ppg_h"].append(pph)
    cols["f_gf_a"].append(gfa); cols["f_ga_a"].append(gaa); cols["f_ppg_a"].append(ppa)

    exp_h = 1.0 / (1.0 + 10 ** (-(eh - ea) / 400.0))
    res_h = 1.0 if hs > asc else (0.5 if hs == asc else 0.0)
    delta = ELO_K * (np.log1p(abs(hs - asc)) + 1.0) * (res_h - exp_h)
    elo[h] = eh + delta; elo[a] = ea - delta

    ph = 3 if hs > asc else (1 if hs == asc else 0)
    pa = 3 if asc > hs else (1 if hs == asc else 0)
    form[h].append((hs, asc, ph)); form[a].append((asc, hs, pa))
    cnt[h] += 1; cnt[a] += 1

for k, v in cols.items():
    df[k] = v
ELO_FINAL = dict(elo)

In [11]:
rk = (fifa[["rank_date","country_full","rank","total_points"]]
      .rename(columns={"rank_date":"date"}).sort_values("date"))
DEF_RANK, DEF_PTS = 150.0, 0.0

df["home_team"] = df["home_team"].astype(str)
df["away_team"] = df["away_team"].astype(str)
rk["country_full"] = rk["country_full"].astype(str)

for side in ["home","away"]:
    r2 = rk.rename(columns={"country_full":f"{side}_team","rank":f"rank_{side[0]}",
                            "total_points":f"pts_{side[0]}"})
    df = pd.merge_asof(df, r2, on="date", by=f"{side}_team", direction="backward")

df["rank_h"] = df["rank_h"].fillna(DEF_RANK); df["rank_a"] = df["rank_a"].fillna(DEF_RANK)
df["pts_h"]  = df["pts_h"].fillna(DEF_PTS);   df["pts_a"]  = df["pts_a"].fillna(DEF_PTS)
df = df.reset_index(drop=True)

# Solução do Data Leakage: Calibração de odds base com perturbação Dirichlet
def elo_to_1x2_calibrated(eh, ea, seed=SEED):
    np.random.seed(seed)
    ph = 1.0 / (1.0 + 10 ** (-(eh - ea) / 400.0))
    pdraw = 0.26 * (1.0 - np.abs(2 * ph - 1.0))
    p1 = ph * (1 - pdraw); p2 = (1 - ph) * (1 - pdraw)
    base_probs = np.vstack([p1, pdraw, p2]).T

    # Adiciona ruído via Dirichlet concentrada para simular ruído das casas de apostas
    calibrated = []
    for bp in base_probs:
        alpha = bp * 50.0  # Concentração estável do prior
        calibrated.append(dirichlet.rvs(alpha)[0])
    res = np.array(calibrated)
    return res[:,0], res[:,1], res[:,2]

p1, px, p2 = elo_to_1x2_calibrated(df["elo_h"].values, df["elo_a"].values)
df["imp_1"], df["imp_x"], df["imp_2"] = p1, px, p2

df["elo_diff"] = df["elo_h"] - df["elo_a"]
df["neutral"]  = df["neutral"].astype(int)
WINDOW_START = DATA_MAX - pd.DateOffset(years=20)
train_df = df[df["date"] >= WINDOW_START].copy()

FEATS = ["elo_h","elo_a","elo_diff","rank_h","rank_a","pts_h","pts_a",
         "f_gf_h","f_ga_h","f_ppg_h","f_gf_a","f_ga_a","f_ppg_a","neutral","imp_1","imp_x","imp_2"]

for c in FEATS:
    if train_df[c].isna().any():
        train_df[c] = train_df[c].fillna(train_df[c].mean())
print("Janela de treino configurada:", len(train_df), "amostras.")

Janela de treino configurada: 19332 amostras.


In [12]:
HALF_LIFE_Y = 2.0
age_y = (DATA_MAX - train_df["date"]).dt.days / 365.25
train_df["w"] = np.power(0.5, age_y / HALF_LIFE_Y)

from sklearn.preprocessing import StandardScaler
X = train_df[FEATS].values.astype("float32")
Y = train_df[["home_score","away_score"]].values.astype("float32")
W = train_df["w"].values.astype("float32")
scaler = StandardScaler().fit(X)
Xs = scaler.transform(X).astype("float32")

## 5. Modelo TensorFlow / Keras

In [13]:
def build_model(n_features, seed=SEED):
    tf.keras.utils.set_random_seed(seed)
    init = keras.initializers.GlorotUniform(seed=seed)
    inp = keras.Input(shape=(n_features,), name="features")

    x = keras.layers.Dense(128, activation="relu", kernel_initializer=init)(inp)
    x = keras.layers.BatchNormalization()(x)
    x = keras.layers.Dropout(0.30, seed=seed)(x)

    x = keras.layers.Dense(64, activation="relu", kernel_initializer=init)(x)
    x = keras.layers.BatchNormalization()(x)
    x = keras.layers.Dropout(0.20, seed=seed)(x)

    x = keras.layers.Dense(32, activation="relu", kernel_initializer=init)(x)
    lam = keras.layers.Dense(2, activation="softplus", kernel_initializer=init, name="lambdas")(x)

    model = keras.Model(inp, lam, name="poisson_score_net")
    model.compile(optimizer=keras.optimizers.Adam(1e-3), loss=keras.losses.Poisson())
    return model

model = build_model(len(FEATS))
model.summary()

Model: "poisson_score_net"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ features (InputLayer)           │ (None, 17)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │         2,304 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 128)            │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 64)             │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lambdas (Dense)                 │ (None, 2)              │            66 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 13,474 (52.63 KB)

 Trainable params: 13,090 (51.13 KB)

 Non-trainable params: 384 (1.50 KB)

In [14]:
val_start = DATA_MAX - pd.DateOffset(months=18)
is_val = (train_df["date"] >= val_start).values
Xtr, Ytr, Wtr = Xs[~is_val], Y[~is_val], W[~is_val]
Xva, Yva, Wva = Xs[is_val],  Y[is_val],  W[is_val]

callbacks = [
    keras.callbacks.EarlyStopping(monitor="val_loss", patience=25, restore_best_weights=True),
    keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=10, min_lr=1e-5),
]
hist = model.fit(Xtr, Ytr, sample_weight=Wtr, validation_data=(Xva, Yva, Wva),
                 epochs=200, batch_size=256, verbose=0, callbacks=callbacks)
print(f"Melhor val_loss obtido: {min(hist.history['val_loss']):.4f}")

Melhor val_loss obtido: 0.5553


## 6. Avaliação e Inferência futura

In [15]:
def score_from_lambda(lam, kmax=5):
    ks = np.arange(0, kmax + 1)
    pmf = poisson.pmf(ks[None, :], np.clip(lam, 1e-6, None)[:, None])
    return pmf.argmax(axis=1).astype(int)

def outcome(gh, ga):
    return np.where(gh > ga, 0, np.where(gh == ga, 1, 2))

lam_va = model.predict(Xva, verbose=0)
gh_net, ga_net = score_from_lambda(lam_va[:,0]), score_from_lambda(lam_va[:,1])
mh, ma = int(round(Ytr[:,0].mean())), int(round(Ytr[:,1].mean()))
gh_bl  = np.full(len(Xva), mh); ga_bl = np.full(len(Xva), ma)

mae_n = (np.abs(gh_net - Yva[:,0]) + np.abs(ga_net - Yva[:,1])).mean() / 2
acc_n = (outcome(gh_net, ga_net) == outcome(Yva[:,0], Yva[:,1])).mean()
print(f"Rede Neural c/ Poisson Calibrada | MAE Gols: {mae_n:.3f} | Acerto 1X2: {acc_n:5.1%}")

Rede Neural c/ Poisson Calibrada | MAE Gols: 0.978 | Acerto 1X2: 60.5%


In [16]:
last_form, last_rank, last_pts, n_recent = {}, {}, {}, {}
recent_cut = DATA_MAX - pd.DateOffset(years=6)
recent_df = df[df["date"] >= recent_cut]
for abrv, name in ABRV2NAME.items():
    last_form[abrv] = form_stats(name)
    n_recent[abrv]  = int(((recent_df["home_team"]==name)|(recent_df["away_team"]==name)).sum())

fifa_latest = fifa.sort_values("rank_date").drop_duplicates("country_abrv", keep="last")
abrv2rank = fifa_latest.set_index("country_abrv")["rank"].to_dict()
abrv2pts  = fifa_latest.set_index("country_abrv")["total_points"].to_dict()
abrv2conf = fifa_latest.set_index("country_abrv")["confederation"].to_dict()

conf_form = defaultdict(list); conf_elo = defaultdict(list)
for abrv, name in ABRV2NAME.items():
    c = abrv2conf.get(abrv, "OTHER")
    gf, ga, ppg = last_form[abrv]
    if not np.isnan(gf): conf_form[c].append((gf, ga, ppg))
    conf_elo[c].append(ELO_FINAL.get(name, ELO_BASE))
glob_form = np.nanmean([v for vs in conf_form.values() for v in vs], axis=0)
def conf_mean_form(c):
    vs = conf_form.get(c, [])
    return np.mean(vs, axis=0) if vs else glob_form

K_SHRINK = 8.0
def team_features(abrv):
    name = ABRV2NAME[abrv]; c = abrv2conf.get(abrv, "OTHER")
    elo_t = ELO_FINAL.get(name, ELO_BASE)
    gf, ga, ppg = last_form[abrv]
    cgf, cga, cppg = conf_mean_form(c)
    n = n_recent[abrv]
    if np.isnan(gf): gf, ga, ppg = cgf, cga, cppg; n = 0
    sh = lambda v, cv: (n * v + K_SHRINK * cv) / (n + K_SHRINK)
    gf, ga, ppg = sh(gf, cgf), sh(ga, cga), sh(ppg, cppg)
    rank = float(abrv2rank.get(abrv, DEF_RANK)); pts = float(abrv2pts.get(abrv, DEF_PTS))
    return dict(elo=elo_t, rank=rank, pts=pts, gf=gf, ga=ga, ppg=ppg, n=n_recent[abrv])

In [17]:
def build_match_row(r):
    A, B = team_features(r["A"]), team_features(r["B"])
    neutral = 0 if r["A_sede"] else 1
    return [A["elo"], B["elo"], A["elo"]-B["elo"], A["rank"], B["rank"], A["pts"], B["pts"],
            A["gf"], A["ga"], A["ppg"], B["gf"], B["ga"], B["ppg"],
            neutral, r["imp_1"], r["imp_x"], r["imp_2"]]

Xfx = np.array([build_match_row(r) for _, r in cal.iterrows()], dtype="float32")
Xfx_s = scaler.transform(Xfx).astype("float32")
lam_fx = model.predict(Xfx_s, verbose=0)
cal["lam_A"] = lam_fx[:,0]; cal["lam_B"] = lam_fx[:,1]
cal["gols_A"] = score_from_lambda(lam_fx[:,0])
cal["gols_B"] = score_from_lambda(lam_fx[:,1])

## 7. Exportação e Validação Final do JSON

In [18]:
NOME, TURMA = "Bruno Aires", "2º BIM 2026"
resultados = {}
for _, r in cal.sort_values("jogo").iterrows():
    resultados[f"jogo{int(r['jogo'])}"] = {
        r["A"]: {"gols": int(r["gols_A"])},
        r["B"]: {"gols": int(r["gols_B"])},
    }
saida = {"nome": NOME, "turma": TURMA, "resultados": resultados}

OUT = "bolao_resultado.json"
with open(OUT, "w", encoding="utf-8") as f:
    json.dump(saida, f, ensure_ascii=False, indent=2)
print("JSON salvo com sucesso em:", OUT)

JSON salvo com sucesso em: bolao_resultado.json
